El paquete de _Python_ [scikit-learn](http://scikit-learn.org) (_sklearn_ en lo que sigue) proporciona un marco de trabajo para el aprendizaje automático.

# Aprendizaje supervisado

Para ilustrar el concepto de aprendizaje supervisado vamos a usar el conjunto de datos [_Car Evaluation_](http://archive.ics.uci.edu/ml/datasets/Car+Evaluation) del repositorio [UCI](http://archive.ics.uci.edu/ml/). Este conjunto de datos contiene información acerca de la idoneidad de una serie de coches, en función de los siguientes atributos:
* _WS1_: intensidad de la señal del router 1 en dBm. Valores entre `-74` y `-10`.
* _WS2_: intensidad de la señal del router 2 en dBm. Valores entre `-74` y `-45`.
* _WS3_: intensidad de la señal del router 3 en dBm. Valores entre `-73` y `-40`.
* _WS4_: intensidad de la señal del router 4 en dBm. Valores entre `-77` y `-11`.
* _WS5_: intensidad de la señal del router 5 en dBm. Valores entre `-89` y `-36`.
* _WS6_: intensidad de la señal del router 6 en dBm. Valores entre `-97` y `-61`.
* _WS7_: intensidad de la señal del router 7 en dBm. Valores entre `-98` y `-63`.

La habitación en la que se encontraba el dispositivo se indica mediante el atributo _Room_, que puede tomar los valores `1`, `2`, `3` o `4`.

Para leer los datos desde el fichero `cars.csv` que se proporciona se pueden evaluar las siguientes expresiones ([_Pandas_](http://pandas.pydata.org/) y [_NumPy_](http://www.numpy.org/) son paquetes de _Python_ para análisis de datos y cálculo científico, respectivamente):

In [1]:
# Instalar paquetes pandas y numpy, pip install pandas numpy
import pandas
import numpy

wifi = pandas.read_csv('wifi_localization_excel.csv', header=None, sep=';',
                       names=['WS1', 'WS2', 'WS3', 'WS4', 'WS5', 'WS6', 'WS7', 'Room'])

In [2]:
# Número de filas y columnas
print(wifi.shape)

(2000, 8)


In [3]:
# 10 primeras filas
wifi.head(10)

,WS1,WS2,WS3,WS4,WS5,WS6,WS7,Room
0,-64,-56,-61,-66,-71,-82,-81,1
1,-68,-57,-61,-65,-71,-85,-85,1
2,-63,-60,-60,-67,-76,-85,-84,1
3,-61,-60,-68,-62,-77,-90,-80,1
4,-63,-65,-60,-63,-77,-81,-87,1
5,-64,-55,-63,-66,-76,-88,-83,1
6,-65,-61,-65,-67,-69,-87,-84,1
7,-61,-63,-58,-66,-74,-87,-82,1
8,-65,-60,-59,-63,-76,-86,-82,1
9,-62,-60,-66,-68,-80,-86,-91,1


_sklearn_ no puede trabajar directamente con el conjunto de datos anterior, ya que asume que los valores de las variables discretas están codificadas con números enteros. Para transformar los datos a un formato adecuado ofrece diversas operaciones de preprocesamiento, entre las que se encuentran `OrdinalEncoder`, para codificar los atributos, y `LabelEncoder`, para codificar la variable objetivo.

In [4]:
# Instalar paquete scikit-learn, pip install scikit-learn
from sklearn import preprocessing

atributos = wifi.loc[:, 'WS1':'WS7']  # selección de las columnas de atributos
objetivo = wifi['Room']  # selección de la columna objetivo

# Para realizar una codificación de los datos, se crea una instancia del tipo de
# codificación pretendida y se ajusta a los datos disponibles mediante el método
# fit. Los métodos transform e inverse_transform permiten entonces codificar y
# descodificar, respectivamente, los datos.

In [5]:
# Los atributos ya son numéricos (intensidad de señal en dBm),
# por lo que no es necesario codificarlos con OrdinalEncoder.
atributos_codificados = atributos

In [6]:
#No se usa ya son numericos

# Codificación de los atributos una vez ajustado el codificador
#atributos_codificados = codificador_atributos.transform(atributos)
#print(atributos_codificados)

In [7]:
#No se usa
# Las respuestas predichas por los modelos estarán codificadas, pero el codificador
# también permite descodificarlas
#print(codificador_atributos.inverse_transform([[3., 3., 0., 0., 2., 1.],
 #                                              [1., 1., 3., 2., 0., 1.]]))

In [8]:
# El codificador adecuado para la variable objetivo es LabelEncoder, que trabaja
# con la lista de sus valores, en lugar de con un array.
codificador_objetivo = preprocessing.LabelEncoder()

# El método fit_transform ajusta la codificación y la aplica a los datos a continuación
objetivo_codificado = codificador_objetivo.fit_transform(objetivo)

In [9]:
# Clases detectadas por el codificador para la variable objetivo
print(codificador_objetivo.classes_)

[1 2 3 4]


In [10]:
print(objetivo_codificado)

[0 0 0 ... 3 3 3]


In [11]:
print(codificador_objetivo.inverse_transform([2, 1, 3]))

[3 2 4]


Una vez preparadas las variables, es necesario separar el conjunto de datos en dos: un conjunto de entrenamiento, que se usará para construir los distintos modelos; y un conjunto de prueba, que se usará para comparar los distintos modelos.

En este caso, la distribución de ejemplos en las distintas habitaciones es uniforme: hay 500 mediciones (un 25 % del total) en cada una de las habitaciones `1`, `2`, `3` y `4`.

In [13]:
# Cantidad total de ejemplos
print(wifi.shape[0])

2000


In [15]:
# Frecuencia total de cada habitación
print(pandas.Series(objetivo).value_counts(normalize=True))

Room
1    0.25
2    0.25
3    0.25
4    0.25
Name: proportion, dtype: float64


Es conveniente, por tanto, que la separación de los ejemplos se realice de manera estratificada, es decir, intentando mantener la proporción anterior tanto en el conjunto de entrenamiento como en el de prueba.

Para dividir un conjunto de datos en un subconjunto de entrenamiento y otro de prueba, _sklearn_ proporciona la función `train_test_split`.

In [18]:
from sklearn import model_selection

(atributos_entrenamiento, atributos_prueba,
 objetivo_entrenamiento, objetivo_prueba) = model_selection.train_test_split(
        # Conjuntos de datos a dividir, usando los mismos índices para ambos
        atributos_codificados, objetivo_codificado,
        # Valor de la semilla aleatoria, para que el muestreo sea reproducible,
        # a pesar de ser aleatorio
        random_state=12345,
        # Tamaño del conjunto de prueba
        test_size=.33,
        # Estratificamos respecto a la distribución de valores en la variable objetivo
        stratify=objetivo_codificado)

In [19]:
# Comprobamos que el conjunto de prueba contiene el 33 % de los datos, en la misma proporción
# con respecto a la variable objetivo
print("Cantidad de ejemplos de pruebas requeridos:", 2000 * .33)
print("Filas del array de atributos de prueba:", atributos_prueba.shape[0])
print("Longitud del vector de objetivos de prueba:", len(objetivo_prueba))
print("Proporción de clases en el vector de objetivos de prueba:")
print(pandas.Series(
        codificador_objetivo.inverse_transform(objetivo_prueba)
      ).value_counts(normalize=True))

Cantidad de ejemplos de pruebas requeridos: 660.0
Filas del array de atributos de prueba: 660
Longitud del vector de objetivos de prueba: 660
Proporción de clases en el vector de objetivos de prueba:
4    0.25
1    0.25
2    0.25
3    0.25
Name: proportion, dtype: float64


In [20]:
# Comprobamos que el conjunto de entrenamiento contiene el resto de los datos, en la misma
# proporción con respecto a la variable objetivo
print("Cantidad de ejemplos de entrenamiento requeridos:", 2000 * .67)
print("Filas del array de atributos de entrenamiento:", atributos_entrenamiento.shape[0])
print("Longitud del vector de objetivos de entrenamiento:", len(objetivo_entrenamiento))
print("Proporción de clases en el vector de objetivos de entrenamiento:")
print(pandas.Series(
        codificador_objetivo.inverse_transform(objetivo_entrenamiento)
      ).value_counts(normalize=True))

Cantidad de ejemplos de entrenamiento requeridos: 1340.0
Filas del array de atributos de entrenamiento: 1340
Longitud del vector de objetivos de entrenamiento: 1340
Proporción de clases en el vector de objetivos de entrenamiento:
1    0.25
3    0.25
4    0.25
2    0.25
Name: proportion, dtype: float64


Para realizar aprendizaje supervisado en _sklearn_ basta crear una instancia de la clase de objetos que implemente el modelo que se quiera utilizar (árboles de decisión, _naive_ Bayes, _kNN_, etc.).

Cada una de estas instancias dispondrá de los siguientes métodos:
* El método `fit` permite entrenar el modelo, dados __por separado__ el conjunto de ejemplos de entrenamiento y la clase de cada uno de estos ejemplos.
* El método `predict` permite clasificar un nuevo ejemplo una vez entrenado el modelo.
* El método `score` calcula el rendimiento del modelo, dados __por separado__ el conjunto de ejemplos de prueba y la clase de cada uno de estos ejemplos.

### Árboles de decisión

_sklearn_ implementa los árboles de decisión clasificadores como instancias de la clase `DecisionTreeClassifier`.

Desafortunadamente, son árboles de decisión binarios construidos asumiendo atributos continuos y mediante un algoritmo distinto a _ID3_, que no está implementado.

En http://scikit-learn.org/stable/modules/tree.html se puede encontrar información acerca de los árboles de decisión implementados en _sklearn_.

In [22]:
from sklearn import preprocessing
from sklearn import model_selection
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# En scikit-learn, ID3 se especifica mediante criterion='entropy'.
# Nota: La implementación nativa de sklearn soporta divisiones numéricas/ordinales.
modelo_id3 = DecisionTreeClassifier(
    criterion='entropy',  # Usa Ganancia de Información (Entropía) como en ID3
    random_state=12345
)

# Entrenamiento del modelo
modelo_id3.fit(atributos_entrenamiento, objetivo_entrenamiento)

# 7. Evaluación del modelo
predicciones = modelo_id3.predict(atributos_prueba)

# Métricas de precisión general
precision_general = accuracy_score(objetivo_prueba, predicciones)
print(f"Exactitud (Accuracy) del modelo ID3: {precision_general * 100:.2f}%\n")

# Reporte detallado por clase (Precision, Recall, F1-Score)
etiquetas_clases = [f"Habitación {clase}" for clase in codificador_objetivo.classes_]
print("=== Reporte de Clasificación ===")
print(classification_report(
    objetivo_prueba, 
    predicciones, 
    target_names=etiquetas_clases
))

# 8. Visualización en texto de la estructura del árbol
print("\n=== Reglas del Árbol de Decisión ===")
reglas_arbol = export_text(modelo_id3, feature_names=list(atributos.columns))
print(reglas_arbol[:1000])  # Muestra las primeras reglas generadas

Exactitud (Accuracy) del modelo ID3: 97.73%

=== Reporte de Clasificación ===
              precision    recall  f1-score   support

Habitación 1       0.99      0.98      0.99       165
Habitación 2       0.98      0.96      0.97       165
Habitación 3       0.96      0.98      0.97       165
Habitación 4       0.98      0.99      0.98       165

    accuracy                           0.98       660
   macro avg       0.98      0.98      0.98       660
weighted avg       0.98      0.98      0.98       660


=== Reglas del Árbol de Decisión ===
|--- WS1 <= -54.50
|   |--- WS5 <= -59.00
|   |   |--- WS4 <= -56.50
|   |   |   |--- WS3 <= -55.50
|   |   |   |   |--- class: 0
|   |   |   |--- WS3 >  -55.50
|   |   |   |   |--- WS1 <= -56.50
|   |   |   |   |   |--- WS7 <= -87.50
|   |   |   |   |   |   |--- WS2 <= -55.50
|   |   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |   |--- WS2 >  -55.50
|   |   |   |   |   |   |   |--- class: 3
|   |   |   |   |   |--- WS7 >  -87.50
|   |

In [23]:
# =============================================================================
# 9. Predicción con datos nuevos
# =============================================================================

# Definimos varias mediciones nuevas de señal WiFi (en dBm)
nuevas_mediciones = pandas.DataFrame([
    {'WS1': -63, 'WS2': -58, 'WS3': -62, 'WS4': -66, 'WS5': -73, 'WS6': -84, 'WS7': -85},
    {'WS1': -38, 'WS2': -56, 'WS3': -44, 'WS4': -40, 'WS5': -64, 'WS6': -84, 'WS7': -87},
    {'WS1': -50, 'WS2': -53, 'WS3': -50, 'WS4': -52, 'WS5': -62, 'WS6': -82, 'WS7': -83},
    {'WS1': -62, 'WS2': -57, 'WS3': -63, 'WS4': -64, 'WS5': -46, 'WS6': -87, 'WS7': -86}
])

print("--- Datos nuevos a evaluar ---")
print(nuevas_mediciones)

# Los atributos ya son numéricos, no hace falta codificarlos

# Realizamos la predicción numérica con el modelo ID3
predicciones_numericas = modelo_id3.predict(nuevas_mediciones)

# Convertimos la predicción numérica (0-3) de vuelta a su habitación original (1-4)
predicciones_etiquetas = codificador_objetivo.inverse_transform(predicciones_numericas)

# Mostramos los resultados en un DataFrame claro
resultados = nuevas_mediciones.copy()
resultados['prediccion_habitacion'] = predicciones_etiquetas

print("\n--- Resultados de la predicción ---")
print(resultados)

--- Datos nuevos a evaluar ---
   WS1  WS2  WS3  WS4  WS5  WS6  WS7
0  -63  -58  -62  -66  -73  -84  -85
1  -38  -56  -44  -40  -64  -84  -87
2  -50  -53  -50  -52  -62  -82  -83
3  -62  -57  -63  -64  -46  -87  -86

--- Resultados de la predicción ---
   WS1  WS2  WS3  WS4  WS5  WS6  WS7  prediccion_habitacion
0  -63  -58  -62  -66  -73  -84  -85                      1
1  -38  -56  -44  -40  -64  -84  -87                      2
2  -50  -53  -50  -52  -62  -82  -83                      3
3  -62  -57  -63  -64  -46  -87  -86                      4


_sklearn_ implementa _naive_ Bayes para atributos categóricos mediante instancias de la clase `CategoricalNB`.

In [ ]:
from sklearn import naive_bayes

clasif_NB = naive_bayes.CategoricalNB(alpha=1.0)  # alpha es el parámetro de suavizado
clasif_NB.fit(atributos_entrenamiento, objetivo_entrenamiento)

Las siguientes expresiones muestran las cuentas realizadas y (los logaritmos de) las probabilidades aprendidas por el modelo.

In [ ]:
for clase, cantidad_ejemplos_clase, log_probabilidad_clase in zip(
    codificador_objetivo.classes_, clasif_NB.class_count_, clasif_NB.class_log_prior_):
        print("Cantidad de ejemplos para la clase", clase, ":", cantidad_ejemplos_clase)
        print("Logaritmo de la probabilidad aprendida para la clase", clase, ":", log_probabilidad_clase)

In [ ]:
for i, atributo in enumerate(cars.columns[:-1]):
    print("Cantidad de ejemplos para el atributo", atributo, ":")
    print(pandas.DataFrame(clasif_NB.category_count_[i],
                          columns=codificador_atributos.categories_[i],
                          index=codificador_objetivo.classes_))
    print("Logaritmos de las probabilidades aprendidas para el atributo", atributo, ":")
    print(pandas.DataFrame(clasif_NB.feature_log_prob_[i],
                          columns=codificador_atributos.categories_[i],
                          index=codificador_objetivo.classes_))
    print()

El método `predict` devuelve la clase predicha por el modelo para un nuevo ejemplo y el método `score` la exactitud (_accuracy_) media sobre un conjunto de datos de prueba.

In [ ]:
nuevos_ejemplos = [['vhigh', 'vhigh', '3', 'more', 'big', 'high'],
                   ['high', 'low', '3', '2', 'med', 'med']]
codificador_objetivo.inverse_transform(clasif_NB.predict(codificador_atributos.transform(nuevos_ejemplos)))

In [ ]:
# Calculamos la fracción de clases correctamente predichas para el conjunto de datos de prueba
clasif_NB.score(atributos_prueba, objetivo_prueba)